# Lekcja 10 - Agenci SI w produkcji

W tej lekcji nauczysz się **wzorów produkcyjnych** dla agentów SI z wykorzystaniem Microsoft Agent Framework (Python). Omawiamy:

- **Obserwowalność** — dodawanie pomiarów czasu i logowania do interakcji agenta
- **Ewaluacja** — wykorzystanie agenta ewaluatora do oceniania jakości odpowiedzi
- **Zarządzanie kosztami** — strategie optymalizacji tokenów i wyboru modelu

Scenariusz to **agent podróży**, który pomaga użytkownikom planować wycieczki, z warstwą monitorowania i ewaluacji na wierzchu.


## Ustawienie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Rozważania dotyczące produkcji

Przenoszenie agentów AI z prototypów do produkcji wymaga starannej uwagi trzech filarów:

1. **Obserwowalność** — Potrzebujesz widoczności tego, co agent robi, ile to trwa i z których narzędzi korzysta. Bez śledzenia i logowania debugowanie problemów produkcyjnych jest niemal niemożliwe.

2. **Ewaluacja** — Zautomatyzowane kontrole jakości zapewniają, że odpowiedzi agenta pozostają dokładne, kompletne i pomocne w czasie. Agent ewaluator może oceniać odpowiedzi według zdefiniowanych kryteriów.

3. **Zarządzanie kosztami** — Zużycie tokenów bezpośrednio wpływa na koszt. Strategie takie jak optymalizacja prompta, wybór modelu i cache'owanie pomagają utrzymać wydatki pod kontrolą bez utraty jakości.


## Tworzenie obserwowalnego agenta

Definiujemy narzędzia do podróży i opakowujemy wywołanie agenta w pomiar czasu, aby móc monitorować opóźnienia. W środowisku produkcyjnym zintegrowałbyś to z OpenTelemetry lub podobnym systemem śledzenia.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Wzorce oceny

Powszechnym wzorcem produkcyjnym jest użycie drugiego agenta jako **oceniającego**. Oceniający punktuje odpowiedź głównego agenta według zdefiniowanych kryteriów, takich jak kompletność, dokładność i użyteczność.

Umożliwia to:
- Automatyczne bramki jakości zanim odpowiedzi dotrą do użytkowników
- Wykrywanie regresji, gdy zmieniają się zapytania lub modele
- Ciągły monitoring wydajności agenta w czasie


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie zarządzania kosztami

Kontrola kosztów jest kluczowa dla produkcyjnych agentów AI. Oto kluczowe strategie:

| Strategia | Opis |
|---|---|
| **Optymalizacja promptów** | Zachowaj instrukcje systemowe zwięzłe. Usuń zbędny kontekst, aby zmniejszyć liczbę tokenów wejściowych. |
| **Wybór modelu** | Używaj mniejszych, tańszych modeli (np. GPT-4o-mini) do prostych zadań, takich jak klasyfikacja lub ekstrakcja, a większe modele zachowaj na złożone rozumowanie. |
| **Pamięć podręczna** | Buforuj wyniki narzędzi i często powtarzane zapytania, aby uniknąć zbędnych wywołań API. |
| **Budżety tokenów** | Ustawiaj limity `max_tokens`, aby zapobiec niespodziewanie długim odpowiedziom. |
| **Grupowanie** | Grupuj wiele zapytań użytkowników w jedno wywołanie API, gdy to możliwe. |

W praktyce dobrze sprawdza się podejście wielopoziomowe: kieruj proste zapytania do szybkiego, taniego modelu i eskaluj tylko złożone zapytania do bardziej zaawansowanego.


## Podsumowanie

W tej lekcji nauczyłeś się, jak:

1. **Dodać obserwowalność** do interakcji agenta poprzez pomiar czasu i logowanie, tworząc podstawy do śledzenia i monitorowania.
2. **Automatycznie oceniać odpowiedzi agenta** za pomocą agenta ewaluatora, który ocenia kompletność, dokładność i przydatność.
3. **Zarządzać kosztami** poprzez optymalizację promptów, wybór modeli, cache’owanie i budżety tokenów.

Te wzorce produkcyjne pomagają zapewnić, że twoi agenci AI są niezawodni, mierzalni i opłacalni na dużą skalę.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
